# Notebook 3: Explore Relevant Projects in the Jira Database

By this point, the reliable emotion labels from Notebook 1 have been loaded into PostgreSQL as `jira_comment_emotion` (Notebook 2). Since the emotion CSV files share comment IDs with `jira_issue_comment`, we can join them to add contextual columns (project, author, timestamp) to the emotion labels to build the new contextualized dataset.

Before building that new dataset, we need to get a better understanding of the data we're working with to be compatible with Kaiaulu. This notebook explores the answers to these questions:

1. Do the emotion label IDs link to `jira_issue_comment` (comments), not `jira_issue_report` (issue reports)?
2. How are the labeled comments distributed across communities (Apache, Spring, JBoss, CodeHaus)?
3. Which Apache projects have the most labeled comments, and what are their Jira project keys?
4. What is the date range of labeled comments per project? (Needed to bound Kaiaulu downloads and avoid IP blocks.)

We also explore the `jira_user` join to understand what author context is available.

The answers inform which project keys we target when generating Kaiaulu config files in Notebook 4.

### Prerequisites

Before running this notebook:

1. **Notebook 2 must have been run**: `jira_comment_emotion` must exist in the `jira` PostgreSQL database.
2. **PostgreSQL must be running locally** with the `jira` database loaded.

### Step 1: Import dependencies and connect to PostgreSQL

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

pd.set_option('display.max_rows', None)

### Step 2: Set configuration

Update the PostgreSQL connection details below to match your local setup.

In [ ]:
PG_HOST     = "localhost"
PG_PORT     = 5432
PG_USER     = "postgres"
PG_PASSWORD = "ADD_YOUR_PASSWORD_HERE"
PG_DB       = "jira"

engine = create_engine(f"postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}")
print("Connected to PostgreSQL.")

### Check 1: Do emotion label IDs link to jira_issue_comment, not jira_issue_report?
The emotion CSV files contain a column named `id`. The id column in the emotion CSVs is supposed to be a Jira comment ID, meaning it should join to `jira_issue_comment.id`, not to `jira_issue_report.id`. The concern is whether these IDs actually pointing at comments, or whether their accidentally pointing at matching issue report IDs.

We verify this with three checks:
- How many emotion IDs match rows in `jira_issue_comment`, and does the matched text look like the actual comment content?
- For any IDs that didn't match, are they bot-generated comments (stored in `jira_issue_bot_comment`), or just missing from the database?
- Do any emotion IDs also appear in `jira_issue_report`?


In [ ]:
with engine.connect() as con:
    total_labeled = pd.read_sql(text("""
        SELECT COUNT(*) AS total FROM jira_comment_emotion;
    """), con).iloc[0, 0]

    # Primary join: how many label IDs match jira_issue_comment?
    comment_matches = pd.read_sql(text("""
        SELECT COUNT(*) AS matched_to_jira_issue_comment
        FROM jira_comment_emotion e
        INNER JOIN jira_issue_comment c ON e.comment_id = c.id;
    """), con).iloc[0, 0]

    # Investigate unmatched IDs: which label IDs have no row in jira_issue_comment?
    unmatched = pd.read_sql(text("""
        SELECT e.comment_id, e.group_number
        FROM jira_comment_emotion e
        WHERE e.comment_id NOT IN (SELECT id FROM jira_issue_comment);
    """), con)

    # Of those unmatched IDs, how many appear in jira_issue_bot_comment?
    bot_matches = pd.read_sql(text("""
        SELECT COUNT(*) AS matched_to_bot_comment
        FROM jira_comment_emotion e
        INNER JOIN jira_issue_bot_comment b ON e.comment_id = b.id
        WHERE e.comment_id NOT IN (SELECT id FROM jira_issue_comment);
    """), con).iloc[0, 0]

    # Cross-check: Do any emotion IDs also appear in `jira_issue_report`?
    report_matches = pd.read_sql(text("""
        SELECT COUNT(*) AS matched_to_jira_issue_report
        FROM jira_comment_emotion e
        INNER JOIN jira_issue_report r ON e.comment_id = r.id;
    """), con).iloc[0, 0]

    # Sample: actual comment text to confirm the join works
    sample = pd.read_sql(text("""
        SELECT e.comment_id, e.group_number, e.love, e.joy, e.sadness,
               LEFT(c.comment, 120) AS comment_preview,
               c.creationdate
        FROM jira_comment_emotion e
        INNER JOIN jira_issue_comment c ON e.comment_id = c.id
        LIMIT 5;
    """), con)

unmatched_count = len(unmatched)
truly_missing = unmatched_count - bot_matches

print(f"Total rows in jira_comment_emotion:  {total_labeled}")
print(f"Matched to jira_issue_comment:       {comment_matches}")
print(f"Unmatched:                           {unmatched_count}")
print(f"  of which in jira_issue_bot_comment: {bot_matches} (auto-generated tool comments, excluded from labeling)")
print(f"Cross-check IDs in jira_issue_report: {report_matches}")
print(f"  of which absent from database:      {truly_missing}")
print()
print(f"Cross-check IDs in jira_issue_report: {report_matches}")
print("  This is expected and harmless: both tables use independent integer IDs.")
print("  The same number appearing in both tables does not mean the label points at an issue report.")
print()
print("Unmatched label IDs by group:")
display(unmatched.groupby('group_number').size().reset_index(name='unmatched_count'))
print()
print("Sample rows confirming comment text is reachable via jira_issue_comment:")
display(sample)


### Check 2: How are labeled comments distributed across communities?

The Jira database covers four open source communities: Apache (ASF), Spring, JBoss, and CodeHaus. Each comment's community is stored in `jira_issue_report.repositoryname`. With this check, we count how many of the loaded emotion-labeled comments belong to each community.

In [ ]:
with engine.connect() as con:
    community_dist = pd.read_sql(text("""
        SELECT
            r.repositoryname AS community,
            COUNT(DISTINCT e.comment_id) AS labeled_comment_count
        FROM jira_comment_emotion e
        INNER JOIN jira_issue_comment c ON e.comment_id = c.id
        INNER JOIN jira_issue_report r ON c.issue_report_id = r.id
        GROUP BY r.repositoryname
        ORDER BY labeled_comment_count DESC;
    """), con)

print("Labeled comment distribution by community:")
display(community_dist)

### Check 3: Which Apache projects have the most labeled comments?

Apache (ASF) is the target community for the Kaiaulu pipeline because Apache Jira projects are publicly accessible without authentication. With this check, we rank Apache projects by labeled comment count and extract the Jira project key from the issue key (e.g., `ZOOKEEPER` from `ZOOKEEPER-1139`).

The project key is what Kaiaulu needs to download data. It becomes the `project_key` field in the YAML config files generated in Notebook 4.

In [ ]:
with engine.connect() as con:
    apache_projects = pd.read_sql(text("""
        SELECT
            SPLIT_PART(r.key, '-', 1) AS project_key,
            r.project_name,
            COUNT(DISTINCT e.comment_id) AS labeled_comment_count
        FROM jira_comment_emotion e
        INNER JOIN jira_issue_comment c ON e.comment_id = c.id
        INNER JOIN jira_issue_report r ON c.issue_report_id = r.id
        WHERE r.repositoryname = 'ASF'
        GROUP BY project_key, r.project_name
        ORDER BY labeled_comment_count DESC;
    """), con)

print(f"Apache projects with emotion-labeled comments: {len(apache_projects)}")
display(apache_projects)

### Check 4: What is the date range of labeled comments per Apache project?

Kaiaulu's `download_jira_issues_by_date()` function accepts a date range to limit downloads. Using the exact date range of our labeled comments avoids unnecessary API calls and reduces the risk of hitting Apache's rate limit or triggering an IP block.

This table will be referenced in Notebook 4 when generating config files, and should be passed as `date_lower_bound` / `date_upper_bound` when running `download_jira_issues.Rmd`.

In [ ]:
with engine.connect() as con:
    date_ranges = pd.read_sql(text("""
        SELECT
            SPLIT_PART(r.key, '-', 1)        AS project_key,
            r.project_name,
            COUNT(DISTINCT e.comment_id)     AS labeled_comment_count,
            MIN(c.creationdate)::date        AS min_date,
            MAX(c.creationdate)::date        AS max_date
        FROM jira_comment_emotion e
        INNER JOIN jira_issue_comment c ON e.comment_id = c.id
        INNER JOIN jira_issue_report r ON c.issue_report_id = r.id
        WHERE r.repositoryname = 'ASF'
        GROUP BY project_key, r.project_name
        ORDER BY labeled_comment_count DESC;
    """), con)

print("Date ranges for Apache projects with emotion-labeled comments:")
display(date_ranges)

### Check 5: Explore the `jira_user` join

This query verifies the `jira_user` join and examines what author context is available.

In [ ]:
with engine.connect() as con:
    # Sample: show author fields for a handful of labeled comments
    user_sample = pd.read_sql(text("""
        SELECT
            e.comment_id,
            c.author_id,
            u.name         AS author_name,
            u.nickname     AS author_nickname,
            u.emailaddress AS author_email
        FROM jira_comment_emotion e
        INNER JOIN jira_issue_comment c ON e.comment_id = c.id
        LEFT JOIN jira_user u ON c.author_id = u.id
        LIMIT 10;
    """), con)

    # Coverage: how many labeled Apache comments have a matched author?
    author_coverage = pd.read_sql(text("""
        SELECT
            COUNT(*)                                                      AS total_apache_comments,
            COUNT(u.id)                                                   AS with_author_match,
            COUNT(*) - COUNT(u.id)                                        AS missing_author,
            ROUND(100.0 * COUNT(u.id) / NULLIF(COUNT(*), 0), 1)          AS match_pct
        FROM jira_comment_emotion e
        INNER JOIN jira_issue_comment c ON e.comment_id = c.id
        INNER JOIN jira_issue_report r  ON c.issue_report_id = r.id
        LEFT JOIN jira_user u           ON c.author_id = u.id
        WHERE r.repositoryname = 'ASF';
    """), con)

print("Sample of author fields for labeled comments:")
display(user_sample)

print("\nAuthor match coverage for Apache labeled comments:")
display(author_coverage)

### Step 6: Build the contextualized dataset
With the join path confirmed, we now attach each emotion label to its full Jira context to build the contextualized dataset.

In [ ]:
contextualized_query = """
SELECT
    e.comment_id,
    e.group_number,
    e.love,
    e.joy,
    e.sadness,
    e.anger,
    e.surprise,
    e.fear,
    c.comment,
    c.creationdate,
    r.project_name,
    r.repositoryname,
    r.key                        AS issue_key,
    SPLIT_PART(r.key, '-', 1)    AS project_key,
    u.name                       AS author_name,
    u.nickname                   AS author_nickname,
    u.emailaddress               AS author_email
FROM jira_comment_emotion e
INNER JOIN jira_issue_comment c ON e.comment_id = c.id
INNER JOIN jira_issue_report r  ON c.issue_report_id = r.id
LEFT JOIN  jira_user u          ON c.author_id = u.id
WHERE r.repositoryname = 'ASF'
ORDER BY e.group_number, e.comment_id;
"""

with engine.connect() as con:
    contextualized = pd.read_sql(text(contextualized_query), con)

print(f"Contextualized Apache dataset: {len(contextualized)} rows, {len(contextualized.columns)} columns")
print(f"Unique comment IDs: {contextualized['comment_id'].nunique()}")
print(f"Projects: {contextualized['project_key'].nunique()}")
display(contextualized.head(10))

### Step 7: Compare original vs. contextualized dataset

Let's see a quick before/after to see what columns we added. The original `jira_comment_emotion` table has 8 columns (comment_id, group_number, and 6 emotion scores). The contextualized dataset has these columns needed by Kaiaulu: comment text, timestamp, project identity, issue key, and author.

In [ ]:
with engine.connect() as con:
    original = pd.read_sql(text("""
        SELECT * FROM jira_comment_emotion LIMIT 3;
    """), con)

print(f"Original jira_comment_emotion — {len(original.columns)} columns:")
print(list(original.columns))
display(original)

print(f"\nContextualized dataset — {len(contextualized.columns)} columns:")
print(list(contextualized.columns))
display(contextualized.head(3))